# SPRKD Quickstart Notebook

This is the **modern, package-based** entry point for SPRKD. It mirrors the end-to-end pipeline of the original `SPRKD.ipynb` but uses the public `sprkd` package APIs and runs unchanged on CPU / CUDA / Apple Silicon (MPS).

For the unmodified ISEF 2023 implementation see [`SPRKD.ipynb`](./SPRKD.ipynb).

## Pipeline

1. Configure the malaria DataLoader.
2. Train a weak teacher with saddle tracking.
3. Aggregate the recorded saddles into an Approximated Saddle Region (ASR).
4. Inject the ASR into the 4x-smaller student via `simple_inject`.
5. Run the SPRKD student loop (Transformation Matrix -> NHE -> PGD).
6. Compare against the released checkpoints on `TESTSET.pth`.

In [1]:
import torch
import torch.nn as nn

from sprkd import (
    MalariaTeacherCNN, MalariaStudentCNN,
    aggregate_asr, simple_inject, set_seed,
    load_legacy_student,
)
from sprkd.data import MalariaDataConfig, find_default_root, make_dataloaders
from sprkd.training import train_teacher, train_student
from sprkd.utils import get_device

set_seed(0)
device = get_device()
print('device:', device)

device: mps


## 1. DataLoader

In [2]:
cfg = MalariaDataConfig(
    root=find_default_root(),
    batch_size=64,
    num_workers=0,
    seed=0,
)
train_loader, valid_loader, full = make_dataloaders(cfg)
print('total cells:', len(full))

total cells: 27558


## 2. Train the (weak) teacher with saddle tracking

In [3]:
teacher = MalariaTeacherCNN().to(device)
sprkd_t, t_history = train_teacher(
    teacher, train_loader, valid_loader,
    loss_fn=nn.CrossEntropyLoss(),
    n_epochs=2,
    saddle_steps=1,
    device=device,
)
print(f'recorded {len(sprkd_t.saddle_repository)} saddles')
print(f'best teacher val acc: {t_history.best_valid_acc():.2f}%')

train:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/adityadewan/anaconda3/envs/sprkd/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/adityadewan/anaconda3/envs/sprkd/lib/python3.10/site-packages/torch/autograd/graph.py:882: UserWarning: Using backward() with create_graph=True will create a reference cycle between the parameter and its gradient which can cause a memory leak. We recommend using autograd.grad when creating the graph to avoid this. If you have to use this function, make sure to reset the .grad fields of your parameters to None after use to break the cycle and avoid the leak. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/engine.cpp:1304.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


recorded 10 saddles
best teacher val acc: 73.33%


## 3. Build the ASR + inject into the student

In [4]:
if len(sprkd_t.saddle_repository) == 0:
    sprkd_t.saddle_repository.append(list(teacher.parameters()), loss=0.0)
asr = aggregate_asr([sprkd_t.saddle_repository.snapshots])
print('ASR layers:', len(asr))

student = MalariaStudentCNN().to(device)
simple_inject(student, teacher)
targets = [p.detach().clone() for p in student.parameters()]
print('student params:', sum(p.numel() for p in student.parameters()))

ASR layers: 8
student params: 6430


## 4. SPRKD student training (Transformation Matrix -> NHE -> PGD)

In [5]:
sprkd_s, s_history = train_student(
    student, train_loader, valid_loader,
    loss_fn=nn.CrossEntropyLoss(),
    teacher_saddle_points=targets,
    n_epochs=2,
    device=device,
)
print(f'best SPRKD student val acc: {s_history.best_valid_acc():.2f}%')

train:   0%|          | 0/2 [00:00<?, ?it/s]

best SPRKD student val acc: 76.21%


## 5. Compare against the released ISEF 2023 checkpoints

The `MODELS/` directory contains the original released checkpoints. Loading them through `sprkd.load_legacy_student` extracts the underlying `nn.Module` from the legacy fastai `Learner` pickles, side-stepping the `__main__.SPRKD` import error.

In [6]:
from pathlib import Path

MODELS = {
    'SPRKD':   Path('..') / 'MODELS' / 'SPRKD_MALARIA.pth',
    'Control': Path('..') / 'MODELS' / 'CONTROL_MALARIA.pth',
    'RKD':     Path('..') / 'MODELS' / 'RKD_MALARIA_STUDENT.pth',
}

ts = torch.load(Path('..') / 'TESTSET.pth', map_location='cpu', weights_only=False)
xs, ys = ts[0].to(device), ts[1].to(device)
print('held-out test set size:', xs.shape[0])

for label, path in MODELS.items():
    if not path.is_file():
        print(f'  {label}: MISSING ({path})')
        continue
    m = load_legacy_student(path).to(device).eval()
    with torch.no_grad():
        preds = torch.argmax(m(xs), dim=1)
    acc = 100.0 * (preds == ys).float().mean().item()
    print(f'  {label}: {acc:.2f}%')

held-out test set size: 100


  SPRKD: 84.00%
  Control: 80.00%
  RKD: 74.00%
